# 2. Feature Engineering

Objetivo: construir un set de variables robusto para predecir `track_popularity`, usando las conclusiones del EDA.

Este notebook no entrena el modelo final. Su tarea es transformar train y test de manera consistente, evitar leakage en encodings supervisados y guardar bases procesadas listas para modelado.


## 1. Imports y rutas


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
from pandas.api.types import is_numeric_dtype

RANDOM_STATE = 42
TARGET = "track_popularity"
ID_COL = "Unnamed: 0"

cwd = Path.cwd()
if cwd.name.lower() in {"codigo", "codigos"}:
    PROJECT_ROOT = cwd.parent.parent if cwd.parent.name.lower() == "soporte" else cwd.parent
elif cwd.name.lower() == "soporte":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd
DATA_DIR = PROJECT_ROOT / "datos_raw"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "bases de datos"
OUTPUT_DIR = PROJECT_ROOT / "soporte" / "otros csv"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Datos: {DATA_DIR}")
print(f"Salida: {OUTPUT_DIR}")


## 2. Carga de datos


In [ ]:
train_raw = pd.read_csv(DATA_DIR / "base_train.csv")
test_raw = pd.read_csv(DATA_DIR / "base_val.csv")

print("Train raw:", train_raw.shape)
print("Test raw:", test_raw.shape)

assert TARGET in train_raw.columns, "Train debe contener track_popularity"
assert TARGET not in test_raw.columns, "Test no deberia contener track_popularity"
assert ID_COL in train_raw.columns and ID_COL in test_raw.columns, "Debe existir columna ID/Unnamed: 0"


## 3. Funciones auxiliares

Se centralizan las funciones para que el mismo proceso pueda aplicarse a train y test sin diferencias accidentales.


In [ ]:
def normalize_text_columns(df):
    """Limpia columnas textuales sin usar informacion del target."""
    out = df.copy()
    for col in out.columns:
        if not is_numeric_dtype(out[col]):
            out[col] = (
                out[col]
                .astype("string")
                .fillna("missing")
                .str.strip()
                .replace({"": "missing", "nan": "missing", "None": "missing", "null": "missing"})
                .astype(str)
            )
    return out


def safe_divide(a, b):
    return np.where(np.abs(b) > 1e-9, a / b, 0)


def add_date_features(df):
    out = df.copy()
    raw = out["track_album_release_date"].astype(str)
    extracted_year = pd.to_numeric(raw.str.extract(r"(\d{4})", expand=False), errors="coerce")
    parsed = pd.to_datetime(raw, errors="coerce", format="mixed")

    out["release_year"] = extracted_year
    out["release_month"] = parsed.dt.month
    out["release_day"] = parsed.dt.day
    out["release_dayofweek"] = parsed.dt.dayofweek
    out["release_decade"] = (extracted_year // 10) * 10
    out["release_age_2020"] = 2020 - extracted_year
    out["release_year_missing"] = extracted_year.isna().astype(int)
    out["release_is_year_only"] = raw.str.match(r"^\d{4}$").astype(int)
    out["release_is_year_month"] = raw.str.match(r"^\d{4}-\d{2}$").astype(int)
    out["release_is_full_date"] = raw.str.match(r"^\d{4}-\d{2}-\d{2}$").astype(int)
    out["release_is_partial"] = ((out["release_is_year_only"] == 1) | (out["release_is_year_month"] == 1)).astype(int)
    out["release_is_recent_2015"] = (extracted_year >= 2015).astype(int)
    out["release_is_old_pre_2000"] = (extracted_year < 2000).astype(int)
    return out


def add_audio_features(df):
    out = df.copy()

    out["duration_min"] = out["duration_ms"] / 60000
    out["duration_sec"] = out["duration_ms"] / 1000
    out["log_duration_ms"] = np.log1p(out["duration_ms"])
    out["duration_short_flag"] = (out["duration_ms"] < 120000).astype(int)
    out["duration_long_flag"] = (out["duration_ms"] > 300000).astype(int)
    out["duration_very_long_flag"] = (out["duration_ms"] > 420000).astype(int)

    out["tempo_zero_flag"] = (out["tempo"] <= 1).astype(int)
    out["tempo_slow_flag"] = (out["tempo"] < 90).astype(int)
    out["tempo_fast_flag"] = (out["tempo"] > 140).astype(int)
    out["tempo_log"] = np.log1p(out["tempo"].clip(lower=0))

    out["energy_x_danceability"] = out["energy"] * out["danceability"]
    out["valence_x_danceability"] = out["valence"] * out["danceability"]
    out["energy_x_loudness"] = out["energy"] * out["loudness"]
    out["acoustic_x_instrumental"] = out["acousticness"] * out["instrumentalness"]
    out["speech_x_energy"] = out["speechiness"] * out["energy"]
    out["dance_energy_valence"] = out["danceability"] * out["energy"] * out["valence"]
    out["energy_minus_acousticness"] = out["energy"] - out["acousticness"]
    out["danceability_minus_speechiness"] = out["danceability"] - out["speechiness"]
    out["valence_minus_energy"] = out["valence"] - out["energy"]
    out["loudness_abs"] = out["loudness"].abs()
    out["loudness_per_energy"] = safe_divide(out["loudness"], out["energy"])
    out["tempo_per_duration_min"] = safe_divide(out["tempo"], out["duration_min"])

    for col in ["speechiness", "acousticness", "instrumentalness", "liveness"]:
        out[f"{col}_log1p"] = np.log1p(out[col])

    out["is_instrumental_high"] = (out["instrumentalness"] > 0.5).astype(int)
    out["is_instrumental_any"] = (out["instrumentalness"] > 0.05).astype(int)
    out["is_acoustic_high"] = (out["acousticness"] > 0.7).astype(int)
    out["is_speech_high"] = (out["speechiness"] > 0.33).astype(int)
    out["is_live_high"] = (out["liveness"] > 0.8).astype(int)
    out["is_positive_high"] = (out["valence"] > 0.75).astype(int)
    out["is_dance_high"] = (out["danceability"] > 0.75).astype(int)
    out["is_energy_high"] = (out["energy"] > 0.8).astype(int)

    return out


def add_categorical_combinations(df):
    out = df.copy()
    out["key_str"] = out["key"].astype(str)
    out["mode_str"] = out["mode"].astype(str)
    out["key_mode"] = out["key_str"] + "_" + out["mode_str"]
    out["genre_subgenre"] = out["playlist_genre"] + "__" + out["playlist_subgenre"]
    out["artist_genre"] = out["track_artist"] + "__" + out["playlist_genre"]
    out["album_genre"] = out["track_album_id"] + "__" + out["playlist_genre"]
    out["playlist_context"] = out["playlist_id"] + "__" + out["playlist_subgenre"]
    return out


def add_unsupervised_frequency_features(train_df, test_df, cols):
    """Agrega frecuencias usando solo X de train+test. No usa target."""
    train_out = train_df.copy()
    test_out = test_df.copy()
    combined = pd.concat([train_out[cols], test_out[cols]], axis=0, ignore_index=True)

    for col in cols:
        all_counts = combined[col].value_counts(dropna=False)
        train_counts = train_out[col].value_counts(dropna=False)

        train_out[f"{col}_freq_all"] = train_out[col].map(all_counts).fillna(0).astype(float)
        test_out[f"{col}_freq_all"] = test_out[col].map(all_counts).fillna(0).astype(float)

        train_out[f"{col}_freq_train"] = train_out[col].map(train_counts).fillna(0).astype(float)
        test_out[f"{col}_freq_train"] = test_out[col].map(train_counts).fillna(0).astype(float)

        train_out[f"{col}_freq_all_log"] = np.log1p(train_out[f"{col}_freq_all"])
        test_out[f"{col}_freq_all_log"] = np.log1p(test_out[f"{col}_freq_all"])

        train_out[f"{col}_seen_in_train"] = train_out[col].isin(train_counts.index).astype(int)
        test_out[f"{col}_seen_in_train"] = test_out[col].isin(train_counts.index).astype(int)

    return train_out, test_out


def add_oof_target_encoding(train_df, test_df, cols, target=TARGET, n_splits=5, smoothing=20, random_state=RANDOM_STATE):
    """Target encoding suavizado. En train se calcula out-of-fold para evitar leakage fila a fila."""
    train_out = train_df.copy()
    test_out = test_df.copy()
    global_mean = train_out[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for col in cols:
        te_col = f"{col}_te_oof"
        train_out[te_col] = np.nan

        for tr_idx, val_idx in kf.split(train_out):
            fold_train = train_out.iloc[tr_idx]
            stats = fold_train.groupby(col)[target].agg(["mean", "count"])
            smooth = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
            train_out.loc[train_out.index[val_idx], te_col] = train_out.iloc[val_idx][col].map(smooth)

        train_out[te_col] = train_out[te_col].fillna(global_mean)

        full_stats = train_out.groupby(col)[target].agg(["mean", "count"])
        full_smooth = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
        test_out[te_col] = test_out[col].map(full_smooth).fillna(global_mean)

        # Version adicional: desvio respecto de la media global.
        train_out[f"{col}_te_diff_global"] = train_out[te_col] - global_mean
        test_out[f"{col}_te_diff_global"] = test_out[te_col] - global_mean

    return train_out, test_out


## 4. Limpieza base y features deterministicas


In [ ]:
train_fe = normalize_text_columns(train_raw)
test_fe = normalize_text_columns(test_raw)

for func in [add_date_features, add_audio_features, add_categorical_combinations]:
    train_fe = func(train_fe)
    test_fe = func(test_fe)

print("Train despues de features deterministicas:", train_fe.shape)
print("Test despues de features deterministicas:", test_fe.shape)


## 5. Frequency encoding

Estas variables no usan `track_popularity`. Miden cuantas veces aparece cada categoria en train y en el conjunto train+test. Son utiles para variables de alta cardinalidad y para categorias nuevas en test.


In [ ]:
freq_cols = [
    "track_id", "track_name", "track_artist", "track_album_id", "track_album_name",
    "playlist_name", "playlist_id", "playlist_genre", "playlist_subgenre",
    "key_str", "mode_str", "key_mode", "genre_subgenre",
    "artist_genre", "album_genre", "playlist_context"
]

train_fe, test_fe = add_unsupervised_frequency_features(train_fe, test_fe, freq_cols)

print("Train despues de frequency encoding:", train_fe.shape)
print("Test despues de frequency encoding:", test_fe.shape)


## 6. Target encoding out-of-fold

Para columnas categoricas con posible senal predictiva se agrega target encoding suavizado. En train se calcula out-of-fold para evitar que una fila use su propio `track_popularity` como feature. En test se usa el mapping aprendido con todo train.

No se target-encodea `track_id` como feature general porque se manejara por separado como lookup final para canciones ya vistas.


In [ ]:
te_cols = [
    "track_artist", "track_album_id", "track_album_name",
    "playlist_name", "playlist_id", "playlist_genre", "playlist_subgenre",
    "key_str", "mode_str", "key_mode", "genre_subgenre",
    "artist_genre", "album_genre", "playlist_context"
]

train_fe, test_fe = add_oof_target_encoding(
    train_fe,
    test_fe,
    cols=te_cols,
    target=TARGET,
    n_splits=5,
    smoothing=20,
    random_state=RANDOM_STATE
)

print("Train despues de target encoding:", train_fe.shape)
print("Test despues de target encoding:", test_fe.shape)


## 7. Lookup de `track_id`

Del EDA surge que cuando un `track_id` aparece repetido en train conserva la misma popularidad. Por eso se guarda una tabla de lookup para usar en la etapa de prediccion final. Esta regla debe aplicarse despues del modelo: si el `track_id` de test fue visto en train, usar su popularidad media; si no, usar la prediccion del modelo.


In [ ]:
track_lookup = (
    train_raw.groupby("track_id")[TARGET]
    .agg(track_id_target_mean="mean", track_id_target_count="count", track_id_target_nunique="nunique")
    .reset_index()
)

seen_test_mask = test_raw["track_id"].isin(set(train_raw["track_id"]))
print("Filas de test con track_id visto en train:", seen_test_mask.sum())
print("Porcentaje:", round(seen_test_mask.mean() * 100, 2), "%")
print("Track_id repetidos con popularidad no constante:", (track_lookup["track_id_target_nunique"] > 1).sum())

track_lookup.head()


## 8. Seleccion y orden final de columnas

Se conservan:

- `ID` para submit.
- `track_popularity` solo en train.
- Categoricas limpias para modelos que soporten categoricas.
- Variables numericas originales y creadas.
- Frequency encoding.
- Target encoding out-of-fold.

No se elimina informacion todavia salvo que sea claramente no predictiva como predictor directo. La seleccion final puede ajustarse en el notebook de modelado.


In [ ]:
# Columnas que se conservan como categoricas limpias para modelos tipo CatBoost.
categorical_model_cols = [
    "track_id", "track_name", "track_artist", "track_album_id", "track_album_name",
    "track_album_release_date", "playlist_name", "playlist_id",
    "playlist_genre", "playlist_subgenre", "key_str", "mode_str", "key_mode",
    "genre_subgenre", "artist_genre", "album_genre", "playlist_context"
]

# Todas las columnas numericas disponibles, excluyendo el target para test.
numeric_model_cols = [
    col for col in train_fe.columns
    if col != TARGET and is_numeric_dtype(train_fe[col])
]

# Evitar usar el ID original como predictor numerico; se conserva aparte como ID.
numeric_model_cols = [col for col in numeric_model_cols if col != ID_COL]

feature_cols = categorical_model_cols + numeric_model_cols
feature_cols = [col for col in feature_cols if col in train_fe.columns and col in test_fe.columns]

train_features = train_fe[[ID_COL, TARGET] + feature_cols].copy()
test_features = test_fe[[ID_COL] + feature_cols].copy()

# Renombrar ID para que ya quede alineado con el formato de entrega.
train_features = train_features.rename(columns={ID_COL: "ID"})
test_features = test_features.rename(columns={ID_COL: "ID"})

print("Cantidad final de features:", len(feature_cols))
print("Train features:", train_features.shape)
print("Test features:", test_features.shape)

assert train_features.drop(columns=[TARGET]).columns.tolist() == test_features.columns.tolist(), "Train/test deben tener las mismas columnas predictoras"


## 9. Control de calidad del set procesado


In [ ]:
quality_summary = pd.DataFrame({
    "base": ["train_features", "test_features"],
    "filas": [len(train_features), len(test_features)],
    "columnas": [train_features.shape[1], test_features.shape[1]],
    "nulos_totales": [train_features.isna().sum().sum(), test_features.isna().sum().sum()],
    "duplicados_ID": [train_features["ID"].duplicated().sum(), test_features["ID"].duplicated().sum()]
})

quality_summary


In [ ]:
# Si quedan nulos numericos por fechas parciales, se imputaran en modelado.
# Aun asi, se deja un resumen para saber donde estan.
missing_processed = pd.DataFrame({
    "missing_train": train_features.isna().sum(),
    "missing_train_%": train_features.isna().mean() * 100,
    "missing_test": test_features.isna().sum(),
    "missing_test_%": test_features.isna().mean() * 100,
}).query("missing_train > 0 or missing_test > 0").sort_values(["missing_train_%", "missing_test_%"], ascending=False)

missing_processed.head(30).round(3)


## 10. Guardado de archivos procesados


In [ ]:
train_features_path = OUTPUT_DIR / "train_features.csv"
test_features_path = OUTPUT_DIR / "test_features.csv"
lookup_path = OUTPUT_DIR / "track_id_lookup.csv"
metadata_path = OUTPUT_DIR / "feature_columns.csv"

train_features.to_csv(train_features_path, index=False)
test_features.to_csv(test_features_path, index=False)
track_lookup.to_csv(lookup_path, index=False)
pd.DataFrame({"feature": feature_cols}).to_csv(metadata_path, index=False)

print("Archivos guardados:")
print(train_features_path)
print(test_features_path)
print(lookup_path)
print(metadata_path)


## 11. Resumen

El set procesado queda listo para el notebook de modelado. Las transformaciones principales fueron:

- limpieza consistente de textos;
- variables temporales desde `track_album_release_date`;
- variables de duracion;
- interacciones musicales;
- flags musicales interpretables;
- combinaciones categoricas;
- frequency encoding;
- target encoding out-of-fold;
- tabla de lookup para `track_id` visto en train.
